# 02. Treinamento e Fine-Tuning do Detector YOLOv8 de Placas Veiculares

Este notebook está **100% pronto para execução no Google Colab com GPU gratuita**.

### Fluxo Automatizado:
1. **Ambiente:** Instalação das dependências e checagem da GPU.
2. **Dados:** Download automático do dataset de placas (`keremberke/license-plate-object-detection`) do Hugging Face e conversão direta para formato YOLO.
3. **Treino:** Fine-tuning do modelo leve `yolov8n.pt` para detecção de placas.
4. **Avaliação:** Cálculo das métricas de teste (mAP@0.5, precisão e recall).
5. **Exportação ONNX:** Conversão com `opset=12` e `simplify=True` para inferência rápida offline em CPU.
6. **Verificação & Download:** Checagem dos tensores de saída e download do arquivo `best.onnx` diretamente para sua máquina.

---
## 1. Instalação de Dependências e Checagem da GPU
*Certifique-se de que a GPU está ativada em: `Ambiente de Execução` -> `Alterar tipo de ambiente de execução` -> `T4 GPU`.*

In [ ]:
!pip install -q ultralytics datasets onnx onnxruntime

import torch
print('Versão do PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('✅ GPU detectada:', torch.cuda.get_device_name(0))
else:
    print('⚠️ ATENÇÃO: Nenhuma GPU detectada. Ative a GPU no menu Ambiente de Execução!')

---
## 2. Download Automático do Dataset e Conversão para Formato YOLO
Baixa o dataset público de detecção de placas e organiza as pastas em `images/` e `labels/` com coordenadas normalizadas.

In [ ]:
import os
from pathlib import Path
from datasets import load_dataset
from tqdm.auto import tqdm

print("Baixando dataset diretamente via Parquet do Hugging Face...")
base_url = (
    "https://huggingface.co/datasets/"
    "keremberke/license-plate-object-detection/"
    "resolve/refs%2Fconvert%2Fparquet/full"
)

# Carrega os arquivos Parquet diretamente (100% compatível com as versões recentes)
ds = load_dataset(
    "parquet",
    data_files={
        "train": f"{base_url}/train/0000.parquet",
        "val": f"{base_url}/validation/0000.parquet",
        "test": f"{base_url}/test/0000.parquet",
    }
)

print("Dataset carregado com sucesso:", ds)

base_dir = Path("/content/dataset")
splits = ["train", "val", "test"]

# Dica: deixe None para exportar as 6.176 imagens de treino,
# ou coloque um número (ex: 2000) se quiser acelerar o primeiro teste
LIMITE_TREINO = None

for split in splits:
    img_dir = base_dir / "images" / split
    lbl_dir = base_dir / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    subset = ds[split]
    if split == "train" and LIMITE_TREINO is not None:
        subset = subset.select(range(min(LIMITE_TREINO, len(subset))))

    print(f"Exportando split {split} ({len(subset)} amostras)...")

    for i, item in enumerate(tqdm(subset)):
        img = item["image"]
        w_img, h_img = img.size

        img_name = f"{split}_{i:05d}.jpg"
        img.save(img_dir / img_name, format="JPEG")

        linhas_yolo = []
        for bbox in item["objects"]["bbox"]:
            xmin, ymin, bw, bh = bbox
            # Coordenadas normalizadas para YOLO: cx, cy, w, h
            cx = (xmin + bw / 2.0) / w_img
            cy = (ymin + bh / 2.0) / h_img
            w_norm = bw / w_img
            h_norm = bh / h_img
            linhas_yolo.append(f"0 {cx:.6f} {cy:.6f} {w_norm:.6f} {h_norm:.6f}")

        lbl_name = f"{split}_{i:05d}.txt"
        with open(lbl_dir / lbl_name, "w", encoding="utf-8") as f:
            f.write("\n".join(linhas_yolo))

# Criação do arquivo de configuração do dataset para o YOLO
yaml_content = """
path: /content/dataset
train: images/train
val: images/val
test: images/test

names:
  0: placa
"""

with open("/content/placas.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content.strip())

print("✅ Dataset estruturado e placas.yaml criado com sucesso!")

---
## 2.1 Aumento de Dados Offline (Opcional: Duplicação do Dataset)
Permite duplicar ou triplicar fisicamente as imagens de treino no disco aplicando perturbações fotométricas realistas (simulação de noite, ruído de sensor, borrão de movimento e contraste).

> [!NOTE]
> Como as alterações são puramente fotométricas (brilho, ruído e nitidez), as coordenadas das placas nos arquivos `.txt` continuam **100% alinhadas e idênticas**.

In [ ]:
import cv2
import random
import shutil
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# =====================================================================
# CONFIGURAÇÃO DE DUPLICAÇÃO FLEXÍVEL
# =====================================================================
DUPLICAR_DATASET = True   # Alterne para True para duplicar ou False para pular
MULTIPLICADOR = 2         # 2 = duplica (1 cópia nova por imagem), 3 = triplica
# =====================================================================

def aplicar_variacao_realista(img: np.ndarray, tipo: int) -> np.ndarray:
    """Gera variações fotométricas típicas de cenários reais de ALPR."""
    im = img.copy()
    if tipo == 0:
        # Cenário 1: Noite / Baixa iluminação com ruído sutil
        gamma = random.uniform(0.50, 0.70)
        inv_gamma = 1.0 / gamma
        tabela = np.array([((i / 255.0) ** inv_gamma) * 255 for i in range(256)]).astype('uint8')
        im = cv2.LUT(im, tabela)
        ruido = np.random.normal(0, 10, im.shape).astype(np.float32)
        im = np.clip(im.astype(np.float32) + ruido, 0, 255).astype(np.uint8)
    elif tipo == 1:
        # Cenário 2: Movimento rápido do veículo (Motion Blur horizontal)
        k = random.choice([5, 7])
        kernel = np.zeros((k, k))
        kernel[int((k - 1) / 2), :] = np.ones(k) / k
        im = cv2.filter2D(im, -1, kernel)
    elif tipo == 2:
        # Cenário 3: Sol intenso / Contraste e brilho elevados
        im = cv2.convertScaleAbs(im, alpha=random.uniform(1.2, 1.4), beta=random.uniform(10, 25))
    else:
        # Cenário 4: Granulação de sensor de segurança (ISO alto)
        ruido = np.random.normal(0, 16, im.shape).astype(np.float32)
        im = np.clip(im.astype(np.float32) + ruido, 0, 255).astype(np.uint8)
    return im

if DUPLICAR_DATASET:
    train_img_dir = Path('/content/dataset/images/train')
    train_lbl_dir = Path('/content/dataset/labels/train')
    
    originais = [f for f in train_img_dir.glob('*.jpg') if not f.stem.endswith(('_aug1', '_aug2'))]
    print(f'⚙️ Gerando variações realistas para {len(originais)} imagens de treino...')
    
    for arq_img in tqdm(originais):
        im = cv2.imread(str(arq_img))
        if im is None:
            continue
        arq_lbl = train_lbl_dir / f'{arq_img.stem}.txt'
        tem_lbl = arq_lbl.exists()
        
        for k in range(1, MULTIPLICADOR):
            tipo = (hash(arq_img.stem) + k) % 4
            im_aug = aplicar_variacao_realista(im, tipo)
            cv2.imwrite(str(train_img_dir / f'{arq_img.stem}_aug{k}.jpg'), im_aug)
            if tem_lbl:
                shutil.copyfile(arq_lbl, train_lbl_dir / f'{arq_img.stem}_aug{k}.txt')
                
    total = len(list(train_img_dir.glob('*.jpg')))
    print(f'✅ Dataset de treino ampliado com sucesso! Total agora: {total} imagens.')
else:
    print('ℹ️ Duplicação offline desativada. Treinando com o conjunto original.')

---
## 3. Treinamento (Fine-Tuning) com YOLOv8n
Usamos a variante nano (`yolov8n.pt`) para obter alta precisão com custo computacional mínimo para inferência rápida no desktop.

In [ ]:
from ultralytics import YOLO

# Carrega pesos pré-treinados
modelo = YOLO('yolov8n.pt')

# Inicia fine-tuning com aumento dinâmico completo na GPU
resultados = modelo.train(
    data='/content/placas.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    # --- Data Augmentation Dinâmico: ---
    degrees=25.0,       # Rotação aleatória de +/- 25° para placas diagonais
    shear=10.0,         # Cisalhamento angular (câmeras de lado/postes)
    perspective=0.001,  # Distorção de perspectiva 3D realista
    hsv_v=0.4,          # Variação forte de brilho (noite/sombra/sol)
    hsv_s=0.5,          # Variação de saturação de cor
    translate=0.1,      # Deslocamento no quadro
    scale=0.5,          # Escala (zoom in / zoom out)
    mosaic=1.0,         # Composição de 4 imagens em 1
    # -----------------------------------
    patience=10,
    project='runs/detect',
    name='treino_placas',
    verbose=True
)

---
## 4. Avaliação das Métricas no Conjunto de Teste

In [ ]:
from pathlib import Path
from ultralytics import YOLO

# Encontra automaticamente o arquivo best.pt gerado pelo treino
candidatos = list(Path("/content").glob("**/treino_placas*/weights/best.pt"))

if not candidatos:
    raise FileNotFoundError("Arquivo best.pt não encontrado. Verifique se o treino terminou.")

caminho_best_pt = str(candidatos[0])
print("Arquivo best.pt encontrado em:", caminho_best_pt)
melhor_modelo = YOLO(caminho_best_pt)
metricas = melhor_modelo.val(data="/content/placas.yaml", split="test")
print("=" * 50)
print("MÉTRICAS NO CONJUNTO DE TESTE")
print("=" * 50)
print(f"mAP@0.5:      {metricas.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metricas.box.map:.4f}")
print(f"Precisão:     {metricas.box.mp:.4f}")
print(f"Recall:       {metricas.box.mr:.4f}")
print("=" * 50)

---
## 5. Exportação para Formato ONNX
Exporta o modelo otimizado para CPU, eliminando a dependência de PyTorch no aplicativo desktop.

In [ ]:
caminho_onnx = melhor_modelo.export(
    format='onnx',
    imgsz=640,
    opset=12,
    simplify=True
)
print('✅ Modelo ONNX exportado em:', caminho_onnx)

---
## 6. Verificação do Modelo com onnxruntime em CPU

In [ ]:
import onnxruntime as ort

# Usa diretamente a variável da célula anterior
sessao = ort.InferenceSession(caminho_onnx, providers=["CPUExecutionProvider"])
entrada = sessao.get_inputs()[0]
saida = sessao.get_outputs()[0]

print("Entrada ONNX:", entrada.name, entrada.shape)
print("Saída ONNX:  ", saida.name, saida.shape)
print("✅ Modelo verificado e 100% compatível!")

---
## 7. Download do Modelo para o Desktop
Ao baixar o arquivo `best.onnx`, renomeie-o para `placas.onnx` e coloque-o na pasta `models/` do projeto no seu computador.

In [ ]:
from google.colab import files

print(f"Iniciando download de: {caminho_onnx}")
files.download(caminho_onnx)